# Station 01 — SFT Basics with Unsloth

> **Objective:** Run Unsloth's Llama-3.1-8B Alpaca notebook end-to-end on free Colab.
> Save LoRA adapter in 3 formats: 16-bit merged, 4-bit quantized, GGUF Q4_K_M.

**Before running:**
1. Set runtime type to T4 GPU (Runtime → Change runtime type → T4 GPU)
2. Add your HF_TOKEN as a Colab secret (left panel → 🔑 → add HF_TOKEN)
3. Run all cells top to bottom

## 1. Install Unsloth

In [ ]:
# Colab-specific install. On Linux local, use: pip install unsloth

!pip install "unsloth[cu121-torch240] @ git+https://github.com/unslothai/unsloth.git"

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 2. Load the model + tokenizer

We use Llama-3.1-8B-Instruct with 4-bit quantization (QLoRA). This fits on a free T4 (16GB VRAM).

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,  # auto-detect
    load_in_4bit = True,
)

## 3. Add LoRA adapters

This is where the magic happens. We're adding thin matrices to ~1% of the model's parameters, and only training those. The base weights stay frozen.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank — try 8, 16, 32, 64, 128. Higher = more capacity, more memory.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,  # scaling factor. Rule of thumb: alpha = r or alpha = 2*r.
    lora_dropout = 0,  # 0 is fine for large models
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 4. Load Alpaca dataset

Alpaca is the classic SFT dataset — 52k instruction-following examples. We use a cleaned subset.

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

## 5. Set up the trainer

TRL's SFTTrainer wraps the HF Trainer with sensible defaults for instruction tuning.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,  # For real training: use 1-3 epochs
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

## 6. Train!

On a free T4, this takes ~30 min for 60 steps. Watch the loss curve — it should drop smoothly.

In [ ]:
trainer_stats = trainer.train()

## 7. Test inference

Before saving, sanity-check that the model produces coherent outputs.

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Write a Python function to compute the Fibonacci sequence.",
        "",
        "",
    )
], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## 8. Save in 3 formats

Push the LoRA adapter to HF Hub, save a 16-bit merged model, and export GGUF for llama.cpp/Ollama.

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

# 1. Push LoRA adapter to HF Hub
model.push_to_hub_gguf("your-username/llama-3-8b-alpaca-lora", tokenizer, token=hf_token)

# 2. Save 16-bit merged locally
model.save_pretrained_merged("outputs/merged-16bit", tokenizer, save_method="merged_16bit")

# 3. Export GGUF Q4_K_M (for llama.cpp / Ollama)
model.save_pretrained_gguf("outputs/gguf-q4-k-m", tokenizer, quantization_method="q4_k_m")

## 9. What to commit to this folder

- This notebook with outputs preserved
- Screenshot of the loss curve (from the trainer_stats object) → `assets/loss_curve.png`
- Screenshot of the sample inference → `assets/sample_output.png`
- Fill in `what-i-built.md`, `what-i-learned.md`, `artifact.md`

Then check all the boxes in `../progress.md` under Station 01. **Only then** may you open Station 02's resources.